# 🚗 Detecció de Matrícules — Pipeline Clàssica de Visió per Computador

Aquesta notebook implementa la pipeline següent:

```
Imatge RGB
    │
    ▼
[1] Preprocessament: gris → Gaussià → CLAHE
    │
    ▼
[2] Sobel vertical → Binarització (Otsu)
    │
    ▼
[3] Closing horitzontal → Opening → Labelling (CC)
    │
    ▼
[4] Filtratge per forma (àrea, aspect ratio, extensió, orientació)
    │
    ▼
Sortida: N bounding boxes (la matrícula sempre dins, + alguns FPs)
```

**Filosofia**: prioritzem el *recall* sobre la *precisió*. Preferim retornar diverses bounding boxes (algunes falsos positius) i garantir que la matrícula real sempre estigui entre les detectades.

> 📁 Estructura de carpetes esperada:
> ```
> ./
> ├── data/
> │   └── raw/         ← posa aquí les teves imatges (.jpg, .png, ...)
> └── notebook.ipynb   ← aquest fitxer
> ```

## 1. Imports i configuració

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Reproducibility: same 10 random images each run
random.seed(32)
np.random.seed(32)

# Make matplotlib figures larger by default
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.cmap'] = 'gray'

print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version:  {np.__version__}')

## 2. Carrega 10 imatges aleatòries de `data/raw/`

Recollim totes les imatges del directori i seleccionem 10 a l'atzar (amb `random.seed(42)` perquè els resultats siguin reproduïbles).

In [ ]:
DATA_DIR = Path('data/raw/')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
N_SAMPLES = 100

# Gather all image paths
all_images = sorted([p for p in DATA_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])
print(f'Found {len(all_images)} images in {DATA_DIR}')

# Sample N_SAMPLES (or all of them if fewer are available)
n_to_pick = min(N_SAMPLES, len(all_images))
selected_paths = random.sample(all_images, n_to_pick)
print(f'\nSelected {len(selected_paths)} images:')
for p in selected_paths:
    print(f'  - {p.name}')

## 3. Fase 1 — Preprocessament

**Objectiu**: portar la imatge a un domini on les vores dels caràcters de la matrícula siguin robustes i destaquin clarament.

- **Escala de grisos**: la informació de color no ens serveix per detectar text negre sobre fons clar.
- **Suavitzat Gaussià**: eliminem soroll d'alta freqüència que generaria falses vores. Apliquem $G_\sigma(x,y) = \frac{1}{2\pi\sigma^2}\,e^{-\frac{x^2+y^2}{2\sigma^2}}$ amb $\sigma$ petit ($\approx 1$).
- **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*): equalitza el contrast localment. És imprescindible perquè les imatges tenen condicions d'il·luminació molt diferents (sol, ombra...).

In [ ]:
def preprocess(img_bgr):
    # Phase 1: grayscale -> Gaussian smoothing -> CLAHE.
    # Returns a single-channel uint8 image, contrast-equalised and denoised.

    # 1.1 Grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 1.2 Gaussian smoothing -- sigma small to keep character edges sharp
    blurred = cv2.GaussianBlur(gray, ksize=(5, 5), sigmaX=1.0)

    # 1.3 CLAHE -- local contrast equalisation
    #     clipLimit:    how aggressive the equalisation is
    #     tileGridSize: local neighbourhood size
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)

    return enhanced

## 4. Fase 2 — Sobel vertical + Binarització (Otsu)

**Per què només el gradient vertical?**

Els caràcters d'una matrícula generen una **densitat anormalment alta de vores verticals** en una regió molt acotada (cada caràcter aporta vores verticals als seus laterals). Aquesta és la **propietat estructural** distintiva que volem explotar.

Apliquem la màscara de Sobel $M_x$ que vam veure al curs:

$$M_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

Després **binaritzem** amb el mètode **d'Otsu**, que tria automàticament el llindar òptim maximitzant la variància inter-classe — no cal triar-lo manualment per cada imatge.

In [ ]:
def sobel_vertical_binary(gray):
    # Phase 2: vertical Sobel + Otsu binarisation.
    # Returns (sobel_abs, binary).

    # Vertical Sobel: detects vertical edges (dx=1, dy=0)
    # ddepth=cv2.CV_16S to keep the sign, then take absolute value
    sobel_x = cv2.Sobel(gray, ddepth=cv2.CV_16S, dx=1, dy=0, ksize=3)
    sobel_abs = cv2.convertScaleAbs(sobel_x)

    # Otsu thresholding -- picks the optimal threshold automatically
    _, binary = cv2.threshold(sobel_abs, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return sobel_abs, binary

## 5. Fase 3 — Morfologia matemàtica + etiquetatge

Ara apliquem morfologia per **agrupar les vores verticals disperses** en blobs compactes.

- **Closing horitzontal** $\phi_B(X) = \varepsilon_B(\delta_B(X))$ amb un **element estructurant rectangular horitzontal** (per exemple $25 \times 5$). Això connecta les vores verticals dels caràcters veïns en un únic blob rectangular → la matrícula es converteix en una taca compacta.

- **Opening** $\gamma_B(X) = \delta_B(\varepsilon_B(X))$ amb un EE petit. Elimina blobs massa fins (línies aïllades de carrosseria, soroll residual).

- **Etiquetatge de components connexos** (connectivitat-8): cada blob esdevé un candidat amb id propi.

In [ ]:
def morphology_and_label(binary):
    # Phase 3: horizontal closing -> opening -> connected-components labelling.
    # Returns (morph, num_labels, labels, stats).

    # Horizontal closing kernel: wide rectangle to merge characters into a blob.
    # Width ~25 px works well for ~400-600 px wide car photos.
    # Increase it for higher-resolution images.
    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 3)) # Original 25, 5
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, close_kernel)

    # Small opening to remove thin spurious structures
    open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    morph = cv2.morphologyEx(closed, cv2.MORPH_OPEN, open_kernel)

    # Connected components with stats -- 8-connectivity
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        morph, connectivity=8
    )

    return morph, num_labels, labels, stats

## 6. Fase 4 — Filtratge per forma

De totes les regions candidates, descartem només les que **clarament no poden ser** una matrícula segons criteris geomètrics. Els llindars són **deliberadament laxes** per no perdre la matrícula real:

| Descriptor | Criteri |
|---|---|
| **Àrea relativa** | Entre 0.1 % i 10 % de la imatge |
| **Relació d'aspecte** $w/h$ | Entre 2 i 7 |
| **Extensió** $\frac{\text{àrea}}{w \cdot h}$ | $> 0.4$ |
| **Mida absoluta mínima** | $w > 40$ i $h > 10$ píxels |

Recorda: aquests filtres **no decideixen quin candidat és la matrícula**. Només eliminen els candidats absurds (la carrosseria sencera, una mota de pols...) i deixen passar diversos blobs plausibles.

In [ ]:
# Shape filter parameters -- kept lax to maximise recall

# 85/100 15 % error
# AREA_RATIO_MIN = 0.001   # 0.1 % of the image
# AREA_RATIO_MAX = 0.10    # 10 % of the image
# ASPECT_RATIO_MIN = 2.0
# ASPECT_RATIO_MAX = 7.0
# EXTENT_MIN = 0.4
# MIN_WIDTH = 40
# MIN_HEIGHT = 10

AREA_RATIO_MIN = 0.001   # 0.1 % of the image
AREA_RATIO_MAX = 0.10    # 10 % of the image
ASPECT_RATIO_MIN = 1.5  # 2.0 abans
ASPECT_RATIO_MAX = 9.0  # 7.0 abans
EXTENT_MIN = 0.25       # 0.4 abans
MIN_WIDTH = 40
MIN_HEIGHT = 10


def filter_by_shape(num_labels, stats, img_shape):
    # Phase 4: keep only candidates whose shape descriptors are plausible.
    # Returns a list of bounding boxes (x, y, w, h).

    H, W = img_shape[:2]
    img_area = H * W
    boxes = []

    # Skip label 0 -- that's the background
    for lbl in range(1, num_labels):
        x, y, w, h, area = stats[lbl]

        # Hard size filter
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            continue

        # Relative area
        area_ratio = area / img_area
        if not (AREA_RATIO_MIN <= area_ratio <= AREA_RATIO_MAX):
            continue

        # Aspect ratio
        aspect = w / float(h)
        if not (ASPECT_RATIO_MIN <= aspect <= ASPECT_RATIO_MAX):
            continue

        # Extent: how much of the bounding box is actually filled by the blob
        extent = area / float(w * h)
        if extent < EXTENT_MIN:
            continue

        boxes.append((int(x), int(y), int(w), int(h)))

    return boxes

## 7. Pipeline completa

Encadenem totes les fases en una sola funció que rep una imatge i retorna les bounding boxes i totes les imatges intermèdies (per poder-les visualitzar).

In [ ]:
def detect_plates(img_bgr):
    # Run the full detection pipeline on a single image.
    # Returns a dict with all intermediate stages and the final boxes.

    # Phase 1
    enhanced = preprocess(img_bgr)

    # Phase 2
    sobel_abs, binary = sobel_vertical_binary(enhanced)

    # Phase 3
    morph, num_labels, labels, stats = morphology_and_label(binary)

    # Phase 4
    boxes = filter_by_shape(num_labels, stats, img_bgr.shape)

    return {
        'enhanced': enhanced,
        'sobel':    sobel_abs,
        'binary':   binary,
        'morph':    morph,
        'boxes':    boxes,
        'n_total':  num_labels - 1,  # discount background
    }

## 8. Funcions de visualització

Dues vistes complementàries:
1. **Vista detallada per imatge**: mostra totes les fases intermèdies (útil per debugging).
2. **Vista resum**: només les bounding boxes finals sobre la imatge original (per veure el resultat ràpidament).

In [ ]:
def show_pipeline_stages(img_bgr, result, title=''):
    # Display all intermediate stages plus the final boxes for a single image.
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(result['enhanced'])
    axes[0, 1].set_title('1. Gray + Gaussian + CLAHE')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(result['sobel'])
    axes[0, 2].set_title('2a. Sobel-x (vertical edges)')
    axes[0, 2].axis('off')

    axes[1, 0].imshow(result['binary'])
    axes[1, 0].set_title('2b. Otsu binarisation')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(result['morph'])
    axes[1, 1].set_title('3. Closing + Opening')
    axes[1, 1].axis('off')

    # Final boxes overlaid on the original image
    axes[1, 2].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in result['boxes']:
        rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                 edgecolor='lime', facecolor='none')
        axes[1, 2].add_patch(rect)
    axes[1, 2].set_title(f"4. Bounding boxes (n={len(result['boxes'])})")
    axes[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

## 9. Executem la pipeline sobre les 10 imatges

Per cada imatge mostrem **totes les etapes intermèdies**. Així pots veure clarament què passa a cada fase i ajustar paràmetres si fa falta.

In [ ]:
# Run the pipeline on every selected image and store the results
results = []
for path in selected_paths:
    img = cv2.imread(str(path))
    if img is None:
        print(f'WARNING: could not read {path.name}')
        continue
    res = detect_plates(img)
    results.append((path, img, res))
    print(f"{path.name:30s} -> {len(res['boxes']):2d} candidates "
          f"(from {res['n_total']:3d} connected components)")

In [ ]:
# Detailed view: every intermediate stage for every image
for path, img, res in results:
    show_pipeline_stages(img, res, title=path.name)

## 10. Vista resum — totes les imatges en una graella

Vista compacta amb només les bounding boxes finals sobre cada imatge. Útil per inspeccionar ràpidament si la matrícula ha quedat detectada a totes les imatges.

In [ ]:
# Compact grid of final detections
n = len(results)
cols = 2
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
axes = np.atleast_2d(axes).flatten()

for ax, (path, img, res) in zip(axes, results):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in res['boxes']:
        rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                 edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f"{path.name} -- {len(res['boxes'])} boxes")
    ax.axis('off')

# Hide unused axes
for ax in axes[len(results):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def parse_annotation(txt_path):
    # Returns a list of (x_min, y_min, x_max, y_max) in pixels.
    boxes = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            # try tab first, fallback to whitespace
            parts = line.split('\t')
            if len(parts) < 5:
                parts = line.split()
            if len(parts) < 5:
                continue
            try:
                x = int(parts[1]); y = int(parts[2])
                w = int(parts[3]); h = int(parts[4])
            except ValueError:
                continue
            if w > 0 and h > 0:
                boxes.append((x, y, x+w, y+h))
    return boxes

def get_txt_path(img_path):
    return img_path.with_suffix('.txt')

In [ ]:
def diagnose_failures(failure_paths):
    """For each failing image, show which phase loses the plate."""
    for path in failure_paths:
        img = cv2.imread(str(path))
        gt = parse_annotation(get_txt_path(path))[0]  # (x1,y1,x2,y2)
        gt_cx = (gt[0] + gt[2]) / 2
        gt_cy = (gt[1] + gt[3]) / 2

        res = detect_plates(img)
        _, _, _, stats = morphology_and_label(res['binary'])

        # Find which CC contains the plate's center
        hit_cc = None
        for lbl in range(1, len(stats)):
            x, y, w, h, area = stats[lbl]
            if x <= gt_cx <= x+w and y <= gt_cy <= y+h:
                hit_cc = (x, y, w, h, area)
                break

        if hit_cc is None:
            print(f'{path.name}: ❌ PHASE 3 -- no blob contains the plate')
        else:
            x, y, w, h, area = hit_cc
            aspect = w / h
            extent = area / (w * h)
            print(f'{path.name}: blob found, w={w} h={h} aspect={aspect:.2f} '
                  f'extent={extent:.2f} -- ', end='')
            # Check which filter killed it
            reasons = []
            if w < MIN_WIDTH: reasons.append(f'w<{MIN_WIDTH}')
            if h < MIN_HEIGHT: reasons.append(f'h<{MIN_HEIGHT}')
            if not (ASPECT_RATIO_MIN <= aspect <= ASPECT_RATIO_MAX):
                reasons.append(f'aspect {aspect:.1f} outside [{ASPECT_RATIO_MIN}, {ASPECT_RATIO_MAX}]')
            if extent < EXTENT_MIN: reasons.append(f'extent {extent:.2f}<{EXTENT_MIN}')
            print('❌ PHASE 4 (' + ', '.join(reasons) + ')' if reasons else '✅ should pass')

In [ ]:
failure_paths = [Path(p) for p in [
    "data/raw/eu7.jpg",
    "data/raw/test_043.jpg",
    "data/raw/eu4.jpg",
]]

diagnose_failures(failure_paths)

## 11. Anàlisi i propers passos

**Què mirar als resultats:**

1. ✅ **La matrícula apareix entre les bounding boxes a totes les imatges?** Aquest és el criteri principal (objectiu de *recall* del 100 %).

2. 📊 **Quants falsos positius generes per imatge?** Si són massa (>10), pots tibar una mica els filtres de forma. Si en queden pocs (1-5), és perfecte per al següent pas.

3. 🔧 **Paràmetres que pots ajustar si cal:**
   - `close_kernel` mida: si la matrícula no es fusiona en un sol blob, augmenta l'amplada. Si es fusiona amb altres elements, redueix-la.
   - `ASPECT_RATIO_MIN/MAX`: les matrícules amb perspectiva poden tenir aspect ratios diferents.
   - `clipLimit` del CLAHE: si la imatge té contrast extrem, baixa'l a 1.5.

**Propers passos (fase 5 de la pipeline):**

- Aplicar un **classificador Viola-Jones / Haar cascade** entrenat per matrícules sobre cada bounding box candidata.
- O bé, entrenar un **classificador HOG + SVM** per fer la mateixa verificació.
- O fer **OCR** dins de cada candidat per quedar-nos amb el que té text amb estructura vàlida de matrícula.